# 09 — External Career-Stage Check on Real Résumé Text (Kaggle)

## Purpose
Test whether the two synthetic findings appear on **real** résumé text:
- **07 (job description):** does the favored career stage climb as the JD goes junior → manager?
- **08 (incumbent profile):** does a reference-composition swap shift the career-stage gradient — and is it,
  as in 08, masked in raw cosine and recoverable under mean-centering?

## What this can and cannot show
The external set has **no true age labels**, so this notebook uses **inferred career-stage proxies**
(stated years of experience, seniority in titles) — noisy, and not a protected attribute. It is an
**external-validity / robustness check**, explicitly secondary to 06–08, and is **small-N**. A weak or null
result here is informative about external validity, not a failure of the controlled experiments.

## Two deliberate design choices
1. **The incumbent reference is built from the external résumés themselves**, not from the synthetic
   reference population. Comparing real résumés to synthetic LLM-generated résumés would conflate the signal
   with a large real-vs-synthetic domain shift; building the reference within the external corpus (as 08 did
   within its corpus) removes that confound.
2. **Career stage is inferred from years/titles only.** Legacy/modern tech are kept as **separate covariates**
   and the tech-stack story is *tested*, not folded into the career-stage label — a "legacy stack" is a
   property of technology, not of age.


## Configuration  (shared src/ libraries — same embedder, cache, and backoff as 07 & 08)

In [1]:
import sys, re, hashlib
from pathlib import Path

# sole purpose: put the repo root on sys.path so `src` imports work
_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src.paths import EXTERNAL_DIR, EXPERIMENTS_DIR, EMBEDDINGS_DIR, TABLES_DIR, FIGURES_DIR, rel

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.fulltext import SCREEN_RATE                    # same screen rate as 07/08 by construction
from src.embedding import Embedder, make_token_counter  # shared embedder/cache/backoff (src/embedding.py)
from src.metrics import wilson_ci, mean_ci

for d in (EXTERNAL_DIR, TABLES_DIR, FIGURES_DIR, EMBEDDINGS_DIR):
    d.mkdir(parents=True, exist_ok=True)

DATASET_PATH = EXTERNAL_DIR / "resume_dataset.csv"
STAGE_ORDER = ["early_career", "mid_career", "senior_career", "very_senior", "unknown"]
KNOWN_STAGES = ["early_career", "mid_career", "senior_career", "very_senior"]
MIN_WORDS = 40

# ---- embedder: one shared implementation with 07/08 (src/embedding.py) -------------------
# Real resumes can be very long, so the shared Embedder also handles per-item truncation
# (max_item_tokens=8000), token-budget request packing (request_token_budget=200k), and
# split-on-error — all defaults. allow_api=False -> cache-only: a bare Run All cannot spend.
# Robustness pass: EMB = Embedder(backend="bge-m3", allow_api=True)  (local; time, not money).
EMB = Embedder(backend="openai", allow_api=False, force_reembed=False)

# Job-description ladder (compact variant; deliberately NOT notebook 07's long-form JDs —
# different text, so it lives here and uses its own cache key, "jd_ladder_09")
JD_LADDER = {
    "junior_swe": "Junior software engineer. Writes and tests code under guidance. Familiar with Python or Java, Git, basic data structures, and debugging. Eager to learn.",
    "mid_swe":    "Software engineer. Designs and ships features independently. Python or Java, SQL, REST APIs, testing, code review, cloud basics.",
    "senior_swe": "Senior software engineer. Owns services end to end. Distributed systems, cloud infrastructure, performance, mentoring, cross-team design.",
    "staff_swe":  "Staff engineer. Sets technical direction across teams. Deep distributed-systems and reliability expertise, architecture, large-scale design.",
    "eng_manager":"Software engineering manager. Leads engineers, owns delivery and reliability, hiring and mentoring, distributed systems, cloud, cross-functional leadership.",
}
print("embedder:", EMB.tag)


embedder: openai__text-embedding-3-small


## 1. Load the external dataset

In [2]:
if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Save the Kaggle CSV at {DATASET_PATH}")
raw = pd.read_csv(DATASET_PATH)
print("raw:", raw.shape, "| columns:", raw.columns.tolist())

# Robust column mapping across the common Kaggle résumé schemas.
ren = {}
for col in raw.columns:
    lc = col.strip().lower()
    if lc in ("text", "resume_text", "resume", "resume_str", "cleaned_resume"): ren[col] = "resume_text"
    elif lc in ("job_title", "title", "position"):                              ren[col] = "job_title"
    elif lc in ("category", "categories"):                                      ren[col] = "category"
ext = raw.rename(columns=ren).copy()
if "resume_text" not in ext.columns:
    raise ValueError(f"No résumé-text column found; columns were {raw.columns.tolist()}. "
                     "Tell me the text column name and I'll map it.")
for c in ("job_title", "category"):
    if c not in ext.columns: ext[c] = ""
ext["resume_text"] = ext["resume_text"].fillna("").astype(str)
ext["job_title"]   = ext["job_title"].fillna("").astype(str)
ext["category"]    = ext["category"].fillna("").astype(str)
ext["external_id"] = [f"ext_{i:06d}" for i in range(len(ext))]
ext["word_count"]  = ext["resume_text"].str.split().str.len()
print("loaded:", ext.shape)

raw: (9000, 3) | columns: ['category', 'job_title', 'Text']
loaded: (9000, 5)


## 2. Quality gate — drop empty/very short résumés and exact duplicates

In [3]:
before = len(ext)
ext = ext[ext["word_count"] >= MIN_WORDS].copy()
n_short = before - len(ext)
before2 = len(ext)
ext = ext.drop_duplicates(subset=["resume_text"]).reset_index(drop=True)
n_dup = before2 - len(ext)
print(f"dropped {n_short} short (<{MIN_WORDS} words) and {n_dup} exact-duplicate résumés")
print(f"kept {len(ext)} résumés; median word count {ext['word_count'].median():.0f}")

dropped 0 short (<40 words) and 3259 exact-duplicate résumés
kept 5741 résumés; median word count 1452


## 3. Word-boundary term matching

Naïve substring matching (`term in text`) miscounts badly: "java" matches
"javascript", "go"/"ml"/"ai"/"rest"/"git" match inside unrelated words. Every term is matched here on
alphanumeric boundaries (so "java" ≠ "javascript", "go" ≠ "going"), while still allowing punctuation
neighbours ("ci/cd", "node.js", "c++", "sr.").

In [4]:
def make_counter(terms):
    pats = [re.compile(r"(?<![a-z0-9])" + re.escape(t.lower()) + r"(?![a-z0-9])") for t in terms]
    def count(text):
        b = str(text).lower()
        return sum(1 for p in pats if p.search(b))
    def present(text):
        b = str(text).lower()
        return any(p.search(b) for p in pats)
    count.present = present
    return count

TECH_TERMS = ["software","developer","engineer","programmer","architect","java","python","javascript",
              "android","ios","backend","frontend","full stack","devops","sre","cloud","aws","azure","gcp",
              "kubernetes","docker","database","sql","etl","hadoop","spark","machine learning","data science",
              "data engineer","qa","quality assurance","security","network administrator","systems engineer"]
SENIOR_TITLE  = ["senior","sr","lead","principal","architect","manager","director","head","vp",
                 "vice president","staff","chief"]
VERY_SENIOR   = ["principal","architect","director","head","vp","vice president","chief","distinguished"]
LEGACY_TECH   = ["cobol","fortran","mainframe","vb6","visual basic","classic asp","struts","ejb","weblogic",
                 "websphere","j2ee","jdk 1.4","jdk 1.5","jdk 1.6","oracle 9i","oracle 10g","sql server 2000",
                 "sql server 2005","sql server 2008","windows xp"]
MODERN_TECH   = ["aws","azure","gcp","kubernetes","docker","terraform","spark","python","golang","react",
                 "typescript","microservices","ci/cd","jenkins","graphql","node.js","pytorch","tensorflow"]

count_tech   = make_counter(TECH_TERMS)
count_senior = make_counter(SENIOR_TITLE)
count_vsen   = make_counter(VERY_SENIOR)
count_legacy = make_counter(LEGACY_TECH)
count_modern = make_counter(MODERN_TECH)
print("matchers built:", len(TECH_TERMS), "tech terms")

matchers built: 34 tech terms


## 4. Filter to technical résumés

In [5]:
ext["filter_text"] = (ext["category"] + " " + ext["job_title"] + " " + ext["resume_text"].str[:2000])
ext["is_technical"] = ext["filter_text"].map(count_tech.present)
tech = ext[ext["is_technical"]].reset_index(drop=True)
print(f"technical résumés: {len(tech)} of {len(ext)}")
if "category" in tech and tech["category"].str.strip().ne("").any():
    print(tech["category"].value_counts().head(12).to_string())

technical résumés: 5514 of 5741
category
Datawarehousing, ETL, Informatica Resumes         881
SQL Developers Resumes                            747
Web Developer Resumes                             706
Business Intelligence, Business Object Resumes    680
Network and Systems Administrators Resumes        655
Business Analyst (BA) Resumes                     582
Project Manager Resumes                           488
Java Developers/Architects Resumes                447
Recruiter Resumes                                 328


## 5. Career-stage proxy  (years/titles only; tech stack kept separate)

In [6]:
def extract_years(text):
    b = str(text).lower()
    vals = []
    for pat in [r"(?:over|more than|nearly|approximately|approx\.?|around)\s+(\d{1,2})\s*\+?\s*years?",
                r"(\d{1,2})\s*\+?\s*years?\s+(?:of\s+)?(?:professional\s+)?experience",
                r"experience\s*(?:of|:)?\s*(\d{1,2})\s*\+?\s*years?",
                r"(\d{1,2})\s*\+\s*years?"]:
        for m in re.finditer(pat, b):
            v = int(m.group(1))
            if 0 <= v <= 45: vals.append(v)
    return float(max(vals)) if vals else np.nan

def infer_stage(row):
    y = row["years_experience"]
    if pd.notna(y):
        if y <= 3:  return "early_career"
        if y <= 10: return "mid_career"
        if y <= 20: return "senior_career"
        return "very_senior"
    title = str(row["job_title"]).lower()
    if count_vsen.present(title):   return "very_senior"
    if count_senior.present(title): return "senior_career"
    return "unknown"   # NB: legacy tech is NOT used to assign stage

tech["years_experience"] = tech["resume_text"].map(extract_years)
tech["legacy_tech_count"] = tech["resume_text"].map(count_legacy)   # covariate, not stage
tech["modern_tech_count"] = tech["resume_text"].map(count_modern)   # covariate, not stage
tech["senior_title"]      = (tech["job_title"] + " " + tech["resume_text"].str[:300]).map(count_senior.present)
tech["career_stage"] = pd.Categorical(tech.apply(infer_stage, axis=1), categories=STAGE_ORDER, ordered=True)

dist = tech["career_stage"].value_counts().reindex(STAGE_ORDER)
dist.to_csv(TABLES_DIR / "ext09_career_stage_counts.csv")
print(dist.to_string())
print(f"\n% with stated years: {tech['years_experience'].notna().mean():.0%}")
known = tech[tech.career_stage.isin(KNOWN_STAGES)]
if len(known) == 0 or dist.reindex(KNOWN_STAGES).max() / max(1, len(known)) > 0.85:
    print("\nWARNING: career-stage proxy is degenerate (one bucket dominates or all unknown); "
          "stage-resolved results below are unreliable.")

career_stage
early_career       80
mid_career       2718
senior_career    1303
very_senior       213
unknown          1200

% with stated years: 65%


## 6. Embedder  (src/embedding.py — shared with 07/08)

In [7]:
count_tokens = make_token_counter()   # tiktoken if available (src/embedding.py)


## 7. Embed external résumés and the JD ladder

In [8]:
Xext = EMB.corpus(tech["resume_text"].fillna(" ").tolist(), key="external_resumes")
emb = {rid: Xext[i] for i, rid in enumerate(tech["external_id"])}
jd_names = list(JD_LADDER)
Xjd = EMB.corpus([JD_LADDER[k] for k in jd_names], key="jd_ladder_09")
jd_vec = {k: Xjd[i] for i, k in enumerate(jd_names)}
tech["n_tokens"] = tech["resume_text"].map(count_tokens)
print("external embedded:", Xext.shape, "| JD rungs:", len(jd_names))

  [cache] openai__text-embedding-3-small__external_resumes__934cad808c6b.npy
  [cache] openai__text-embedding-3-small__jd_ladder_09__f9604f6d00a8.npy
external embedded: (5514, 1536) | JD rungs: 5


## 8. Job-description similarity by career stage  (the 07 analogue)

For each rung of the JD ladder, which inferred career stage is most similar? In 07 the favored band climbed
from early-career toward senior/manager as the JD became more senior. The question is whether any of that
ordering survives on real résumé text.

In [9]:
for k in jd_names:
    tech[f"jd__{k}"] = Xext @ jd_vec[k]

rows = []
for k in jd_names:
    for st in KNOWN_STAGES:
        g = tech[tech.career_stage == st][f"jd__{k}"]
        m, lo, hi = mean_ci(g)
        rows.append({"jd": k, "career_stage": st, "n": len(g), "mean_sim": m})
jd_by_stage = pd.DataFrame(rows)
jd_by_stage.to_csv(TABLES_DIR / "ext09_jd_similarity_by_stage.csv", index=False)
piv = jd_by_stage.pivot(index="jd", columns="career_stage", values="mean_sim").reindex(jd_names)[KNOWN_STAGES]
print(piv.round(4).to_string())
print("\nfavored stage by JD rung:", {k: (piv.loc[k].idxmax() if piv.loc[k].notna().any() else None) for k in jd_names})

career_stage  early_career  mid_career  senior_career  very_senior
jd                                                                
junior_swe          0.4009      0.4150         0.3815       0.3590
mid_swe             0.4322      0.4471         0.4235       0.4185
senior_swe          0.4549      0.4643         0.4690       0.4780
staff_swe           0.4173      0.4240         0.4360       0.4519
eng_manager         0.3974      0.4065         0.4210       0.4342

favored stage by JD rung: {'junior_swe': 'mid_career', 'mid_swe': 'mid_career', 'senior_swe': 'very_senior', 'staff_swe': 'very_senior', 'eng_manager': 'very_senior'}


## 9. Manager-JD screen-in rate by stage (Wilson CIs)

In [10]:
thr = tech["jd__eng_manager"].quantile(1 - SCREEN_RATE)
tech["jd_screened"] = tech["jd__eng_manager"] >= thr
rows = []
for st in KNOWN_STAGES:
    s = tech[tech.career_stage == st]["jd_screened"]
    n = int(len(s)); k = int(s.sum())
    lo, hi = wilson_ci(k, n)                                  # src.metrics (returns lo, hi)
    rows.append({"career_stage": st, "n": n, "screen_rate": (k / n if n else np.nan), "lo": lo, "hi": hi})
jd_screen = pd.DataFrame(rows).set_index("career_stage")
jd_screen.to_csv(TABLES_DIR / "ext09_jd_screen_rates.csv")
print(jd_screen.round(3).to_string())


                  n  screen_rate     lo     hi
career_stage                                  
early_career     80        0.312  0.222  0.421
mid_career     2718        0.437  0.419  0.456
senior_career  1303        0.569  0.542  0.595
very_senior     213        0.704  0.640  0.761


## 9b. Length control for the JD gradient  (mirrors 07 §8)

Does the JD-seniority gradient survive removing résumé length? A pure length/centrality artifact would
disappear once similarity is residualized on token count.

In [11]:
x = tech["n_tokens"].to_numpy(float); y = tech["jd__eng_manager"].to_numpy(float)
coef = np.polyfit(x, y, 2); tech["jd_mgr_resid"] = y - np.polyval(coef, x)
g = tech.groupby("career_stage", observed=True)
lc = pd.DataFrame({
    "mean_tokens":            g["n_tokens"].mean().reindex(KNOWN_STAGES),
    "raw_manager_sim":        g["jd__eng_manager"].mean().reindex(KNOWN_STAGES),
    "length_adj_manager_sim": g["jd_mgr_resid"].mean().reindex(KNOWN_STAGES),
})
lc.to_csv(TABLES_DIR / "ext09_jd_length_control.csv")
print(f"corr(manager JD sim, tokens) = {np.corrcoef(x, y)[0, 1]:.3f}")
print(lc.round(4).to_string())

corr(manager JD sim, tokens) = 0.019
               mean_tokens  raw_manager_sim  length_adj_manager_sim
career_stage                                                       
early_career     1830.8750           0.3974                 -0.0093
mid_career       2246.3753           0.4065                 -0.0015
senior_career    2083.5955           0.4210                  0.0144
very_senior      2106.9765           0.4342                  0.0291


## 10. Incumbent reference swap **within the external corpus**  (the 08 analogue)

Exploratory and small-N. R1 = a "current successful" reference (mid/senior career stage with a modern-leaning
stack); R2 = stage-balanced; applicants disjoint from both. As in 08 this is run **raw and mean-centered**,
because raw cosine on a homogeneous corpus is anisotropy/length-dominated. Treat wide intervals and any null
as genuinely inconclusive — the proxy is noisy and partly circular with the embedding.

In [12]:
pool = tech[tech.career_stage.isin(KNOWN_STAGES)].copy()

def _samp(df, frac, cap, rs):
    # cap each reference at a fraction of the stage so applicants always remain (matters for small sets)
    n = min(cap, max(1, int(frac * len(df)))) if len(df) else 0
    return df.sample(n, random_state=rs) if n else df.iloc[0:0]

R1 = pd.concat([_samp(pool[(pool.career_stage == s) & (pool.modern_tech_count >= pool.legacy_tech_count)], 0.4, 150, 42)
                for s in ["mid_career", "senior_career"]]) if len(pool) else pool.iloc[0:0]
R2 = pd.concat([_samp(pool[pool.career_stage == s], 0.3, 120, 7)
                for s in KNOWN_STAGES]) if len(pool) else pool.iloc[0:0]
ref_ids = set(R1.external_id) | set(R2.external_id)
app = pool[~pool.external_id.isin(ref_ids)].copy()
print(f"R1={len(R1)}  R2={len(R2)}  applicants={len(app)}  (small-N: read with care)")
print("applicants per stage:", app.career_stage.value_counts().reindex(KNOWN_STAGES).to_dict())

ref_comp = pd.DataFrame({
    "R1": R1["career_stage"].value_counts().reindex(KNOWN_STAGES, fill_value=0),
    "R2": R2["career_stage"].value_counts().reindex(KNOWN_STAGES, fill_value=0),
    "applicants": app["career_stage"].value_counts().reindex(KNOWN_STAGES, fill_value=0),
})
ref_comp.to_csv(TABLES_DIR / "ext09_incumbent_reference_composition.csv")
ref_comp

def _centroid(ids, transform=lambda v: v):
    V = np.array([transform(emb[c]) for c in ids if c in emb])
    if len(V) == 0: return None
    c = V.mean(axis=0); return c / (np.linalg.norm(c) or 1.0)

def swap_table(transform, tag):
    cR1, cR2 = _centroid(R1.external_id, transform), _centroid(R2.external_id, transform)
    A = np.array([transform(emb[c]) for c in app.external_id])
    app[f"simR1{tag}"] = A @ cR1; app[f"simR2{tag}"] = A @ cR2
    t = app.groupby("career_stage", observed=True)[[f"simR1{tag}", f"simR2{tag}"]].mean().reindex(KNOWN_STAGES)
    return t

raw_tbl = swap_table(lambda v: v, "_raw")
print("RAW cosine to reference centroids:"); print(raw_tbl.round(4).to_string())

R1=300  R2=327  applicants=3706  (small-N: read with care)
applicants per stage: {'early_career': 56, 'mid_career': 2458, 'senior_career': 1042, 'very_senior': 150}
RAW cosine to reference centroids:
               simR1_raw  simR2_raw
career_stage                       
early_career      0.8145     0.8207
mid_career        0.8312     0.8423
senior_career     0.8260     0.8290
very_senior       0.8148     0.8152


In [13]:
mu = np.mean([emb[c] for c in app.external_id], axis=0)   # anisotropy correction (mirrors 08 §6b)
def _cen(v):
    w = v - mu; return w / (np.linalg.norm(w) or 1.0)
cen_tbl = swap_table(_cen, "_cen")
cen_tbl.to_csv(TABLES_DIR / "ext09_incumbent_swap_centered.csv")
raw_tbl.to_csv(TABLES_DIR / "ext09_incumbent_swap_raw.csv")
print("MEAN-CENTERED cosine to reference centroids:"); print(cen_tbl.round(4).to_string())
print("\ncentered swap effect (R2-R1) by stage:",
      (cen_tbl["simR2_cen"] - cen_tbl["simR1_cen"]).round(4).to_dict())

MEAN-CENTERED cosine to reference centroids:
               simR1_cen  simR2_cen
career_stage                       
early_career      0.0102     0.0196
mid_career       -0.0970    -0.0874
senior_career     0.1010     0.1236
very_senior       0.2006     0.2579

centered swap effect (R2-R1) by stage: {'early_career': 0.0094, 'mid_career': 0.0096, 'senior_career': 0.0225, 'very_senior': 0.0572}


## 10b. Stage-balanced centering  (robustness for §10)

§10's centering subtracts the applicant-pool mean, which is dominated by mid-career résumés, so it can
reward distinctiveness-from-average rather than reference composition. Re-running on a stage-balanced
applicant subsample makes the mean composition-neutral; if the centered gradient flattens, §10's shape was a
pool-imbalance artifact.

In [14]:
nbal = int(app.career_stage.value_counts().reindex(KNOWN_STAGES).min())
app_bal = pd.concat([app[app.career_stage == s].sample(nbal, random_state=11) for s in KNOWN_STAGES])
mu_bal = np.mean([emb[c] for c in app_bal.external_id], axis=0)
def _cenbal(v):
    w = v - mu_bal; return w / (np.linalg.norm(w) or 1.0)
Ab = np.array([_cenbal(emb[c]) for c in app_bal.external_id])
app_bal = app_bal.assign(simR1cb=Ab @ _centroid(R1.external_id, _cenbal),
                         simR2cb=Ab @ _centroid(R2.external_id, _cenbal))
bal = app_bal.groupby("career_stage", observed=True)[["simR1cb", "simR2cb"]].mean().reindex(KNOWN_STAGES)
bal.to_csv(TABLES_DIR / "ext09_incumbent_swap_centered_balanced.csv")
print(f"stage-balanced applicants: {nbal}/stage (n={len(app_bal)})")
print(bal.round(4).to_string())
print("balanced swap effect (R2-R1):", (bal["simR2cb"] - bal["simR1cb"]).round(4).to_dict())

stage-balanced applicants: 56/stage (n=224)
               simR1cb  simR2cb
career_stage                   
early_career   -0.0349   0.0260
mid_career     -0.0778   0.1046
senior_career   0.0297  -0.0006
very_senior     0.0094  -0.0638
balanced swap effect (R2-R1): {'early_career': 0.0609, 'mid_career': 0.1824, 'senior_career': -0.0302, 'very_senior': -0.0732}


## 11. Two-target contrast — JD vs incumbent (centered) by stage

In [15]:
app["jd_pct"]  = app["jd__eng_manager"].rank(pct=True)
app["inc_pct"] = app["simR1_cen"].rank(pct=True)
contrast = app.groupby("career_stage", observed=True)[["jd_pct", "inc_pct"]].mean().reindex(KNOWN_STAGES)
contrast["divergence"] = contrast["jd_pct"] - contrast["inc_pct"]
contrast.to_csv(TABLES_DIR / "ext09_two_target_contrast.csv")
print(contrast.round(3).to_string())

               jd_pct  inc_pct  divergence
career_stage                              
early_career    0.416    0.534      -0.119
mid_career      0.468    0.427       0.041
senior_career   0.559    0.636      -0.077
very_senior     0.652    0.743      -0.091


## 12. Tech stack is not age — testing the legacy-stack story

If "legacy stack" résumés score a certain way, that is a fact about *technology terms*, not about a person's
age. Here we check whether legacy/modern tech counts move similarity **independently of inferred stage**,
rather than asserting an enterprise-overlap explanation.

In [16]:
tech["legacy_tertile"] = pd.qcut(tech["legacy_tech_count"].rank(method="first"), 3, labels=["low","mid","high"])
by_legacy = tech.groupby("legacy_tertile", observed=True).agg(
    n=("external_id", "count"),
    mean_jd_manager=("jd__eng_manager", "mean"),
    mean_legacy_count=("legacy_tech_count", "mean"),
    mean_modern_count=("modern_tech_count", "mean"),
)
by_legacy.to_csv(TABLES_DIR / "ext09_similarity_by_legacy_tertile.csv")
print(by_legacy.round(3).to_string())
print(f"\ncorr(legacy_tech_count, jd__eng_manager) = {tech['legacy_tech_count'].corr(tech['jd__eng_manager']):.3f}")
print(f"corr(modern_tech_count, jd__eng_manager) = {tech['modern_tech_count'].corr(tech['jd__eng_manager']):.3f}")

                   n  mean_jd_manager  mean_legacy_count  mean_modern_count
legacy_tertile                                                             
low             1838            0.404              0.060              0.260
mid             1838            0.407              1.643              0.377
high            1838            0.410              5.041              0.579

corr(legacy_tech_count, jd__eng_manager) = 0.069
corr(modern_tech_count, jd__eng_manager) = 0.255


## 13. Save scored external dataset

In [17]:
out = EXPERIMENTS_DIR / "ext09_external_scored.parquet"
keep = [c for c in tech.columns if c != "filter_text"]
tech[keep].to_parquet(out, index=False)
meta = {
    "embedder": EMB.backend,
    "model": EMB.model,
    "source_csv": DATASET_PATH.name,
    "source_csv_sha256": hashlib.sha256(DATASET_PATH.read_bytes()).hexdigest(),
    "rows": int(len(tech)),
}
out.with_suffix(".meta.json").write_text(json.dumps(meta, indent=2))

print("Saved:")
for f in [
    TABLES_DIR / "ext09_jd_similarity_by_stage.csv",
    TABLES_DIR / "ext09_jd_screen_rates.csv",
    TABLES_DIR / "ext09_jd_length_control.csv",
    TABLES_DIR / "ext09_incumbent_swap_raw.csv",
    TABLES_DIR / "ext09_incumbent_swap_centered.csv",
    TABLES_DIR / "ext09_incumbent_swap_centered_balanced.csv",
    TABLES_DIR / "ext09_two_target_contrast.csv",
    TABLES_DIR / "ext09_similarity_by_legacy_tertile.csv",
    out,
    out.with_suffix(".meta.json"),
]:
    print(" -", rel(f))

Saved:
 - reports/tables/ext09_jd_similarity_by_stage.csv
 - reports/tables/ext09_jd_screen_rates.csv
 - reports/tables/ext09_jd_length_control.csv
 - reports/tables/ext09_incumbent_swap_raw.csv
 - reports/tables/ext09_incumbent_swap_centered.csv
 - reports/tables/ext09_incumbent_swap_centered_balanced.csv
 - reports/tables/ext09_two_target_contrast.csv
 - reports/tables/ext09_similarity_by_legacy_tertile.csv
 - data/experiments/ext09_external_scored.parquet
 - data/experiments/ext09_external_scored.meta.json


## 14. Interpretation & caveats

**The job-description mechanism transfers to real résumé text by inferred career stage; the incumbent mechanism does not cleanly transfer.**

1. **JD-by-stage (§8/§9) mirrors notebook 07 on real résumé text.** The favored career stage climbs as the JD ladder ascends
   (junior/mid JDs favor mid-career résumés, while senior/staff/manager JDs favor the most senior inferred stage), and the
   manager-JD screen-in rate rises monotonically across career stages with non-overlapping Wilson intervals.
   The crossover across rungs (different JDs favor different stages) rules out a pure length/centrality
   artifact, and §9b confirms the gradient survives residualizing on résumé length.
2. **Incumbent swap (§10) is inconclusive on this dataset.** Raw cosine washes out (as in 08 — anisotropy). The
   mean-centered version is confounded by the corpus's heavy mid-career imbalance: the applicant-pool mean is
   essentially the mid-career profile, so centering rewards distinctiveness-from-average (rare senior résumés)
   rather than reference composition; §10b re-runs it on a stage-balanced subsample to expose this. The swap
   effect (R2 − R1) is an order of magnitude smaller than 08's and lacks 08's clean sign reversal, so we do
   not claim external replication of the incumbent mechanism — it remains demonstrated under controlled
   synthetic conditions (06/08) only.
3. **Tech stack is not the driver (§12).** Legacy-tech count is only weakly correlated with manager-JD similarity (flat across tertiles); modern tech is modestly positive. The seniority gradient is role/content
   driven, not an enterprise-terminology artifact.
4. **Caveats.** No ground-truth age (career stage is inferred from years/titles); the technical filter is
   permissive (a few non-core-tech categories remain); cosine is embedder-dependent; the career-stage proxy is
   partly circular with the embedding. This notebook shows the JD-target mechanism generalizes to real text and
   is candid that the incumbent-target mechanism does not cleanly survive on this noisy, imbalanced sample.
